# Aligning two coronal sections of adult mouse brain from MERFISH (squidpy + moscot)

In this notebook, we align two single cell resolution spatial transcriptomics datasets of coronal sections of the adult mouse brain from matched locations with respect to bregma assayed by MERFISH.

The original notebook used STalign with affine-only + manual rotation initialization. Here we use `squidpy` with the `moscot` backend which handles alignment automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load source data

In [ ]:
# Single cell data 1
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz'
df1 = pd.read_csv(fname)

coords_source = np.column_stack([df1['center_x'].values, df1['center_y'].values])
adata_source = ad.AnnData(
    X=np.zeros((len(coords_source), 1)),
    obs=df1,
)
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='source')
ax.legend(markerscale=10)

## Load target data

In [ ]:
# Single cell data 2
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate2_cell_metadata_S2R2.csv.gz'
df2 = pd.read_csv(fname)

coords_target = np.column_stack([df2['center_x'].values, df2['center_y'].values])
adata_target = ad.AnnData(
    X=np.zeros((len(coords_target), 1)),
    obs=df2,
)
adata_target.obsm['spatial'] = coords_target

fig, ax = plt.subplots()
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.2, c='#ff7f0e', label='target')
ax.legend(markerscale=10)

In [ ]:
# Plot overlay before alignment
fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2, label='source')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='target')
ax.legend(markerscale=10)

## Align using moscot (optimal transport)

Moscot handles alignment without needing manual rotation angles or pre-transforms.

In [ ]:
# Align
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

## Visualize results

In [ ]:
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.1, label='source')
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='source aligned')
ax.scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='target')
ax.legend(markerscale=10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 8))
ax[0].scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.1, label='source')
ax[0].scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='target')
ax[0].legend(markerscale=10, loc='lower left')
ax[0].set_title('Before alignment')

ax[1].scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='source aligned')
ax[1].scatter(coords_target[:, 0], coords_target[:, 1], s=1, alpha=0.1, label='target')
ax[1].legend(markerscale=10, loc='lower left')
ax[1].set_title('After alignment')

## Save results

In [ ]:
df_aligned = pd.DataFrame({'aligned_x': aligned[:, 0], 'aligned_y': aligned[:, 1]})
results = pd.concat([df1, df_aligned], axis=1)
results.head()